In [1]:
from sklearn.calibration import CalibratedClassifierCV
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, 
    precision_recall_curve, average_precision_score, confusion_matrix, classification_report
)
from joblib import load
from collections import Counter
import xgboost as xgb
import os
from pathlib import Path
from scipy.stats import randint, uniform, loguniform
from sklearn.model_selection import RandomizedSearchCV

In [2]:

from optuna import visualization


class XGboostCalibrator:
    def __init__(self, 
                 model_path= str,
                 scaler_path= str, 
                 model= None,
                 method= str, 
                 fit_data=pd.DataFrame(), 
                 fit_labels=pd.Series(), 
                 test_data = pd.DataFrame(), 
                 test_labels=pd.Series(),
                 visualize_results=True,
                 cv=5):
        """Initialize the calibrator with model, scaler, data, and method"""
        self.uncalibrated_model = load(model_path) if model is None else model
        self.scaler = load(scaler_path)

        self.fit_data = fit_data
        self.fit_labels = fit_labels
        self.test_data = test_data
        self.test_labels = test_labels
        self.method = method
        self.calibrated_model = None
        self.cv = cv
        self.visualize_results = visualize_results

        # Scale the data
        self.fit_data_scaled  =self.scaler.transform(self.fit_data)
        self.test_data_scaled = self.scaler.transform(self.test_data)
    @staticmethod
    def _sample_data( fit_data, fit_labels, test_data, test_labels, sample_size):
        """Sample a subset of the data for quicker calibration and testing"""
        rng = np.random.default_rng(42)

        sample_indices_fit = rng.choice(fit_data.shape[0], size=sample_size, replace=False)

        fit_data = fit_data[sample_indices_fit]
        fit_labels = fit_labels[sample_indices_fit]

        sample_indices_test = rng.choice(test_data.shape[0], size=sample_size, replace=False)

        test_data = test_data[sample_indices_test]
        test_labels = test_labels[sample_indices_test]
        return fit_data, fit_labels, test_data, test_labels


    def _split_train_val_test(self, X, y, train_size=0.64, val_size=0.16, test_size=0.20, random_state=42, stratify=True):
        """
        Split arrays or matrices into train, validation, and test subsets.

        Parameters:
        -----------
        X : np.ndarray or pd.DataFrame
            Features.
        y : np.ndarray or pd.Series
            Labels.
        train_size : float
            Proportion of the dataset to include in the train split.
        val_size : float
            Proportion of the dataset to include in the validation split.
        test_size : float
            Proportion of the dataset to include in the test split.
        random_state : int
            Random seed.
        stratify : bool
            Whether to stratify splits by y.

        Returns:
        --------
        X_train, X_val, X_test, y_train, y_val, y_test
        """
        assert abs(train_size + val_size + test_size - 1.0) < 1e-6, "Splits must sum to 1.0"

        stratify_y = y if stratify else None

        # First split: train+val vs test
        X_trainval, X_test, y_trainval, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=stratify_y
        )

        # Compute val proportion of trainval
        val_prop = val_size / (train_size + val_size)
        stratify_trainval = y_trainval if stratify else None

        # Second split: train vs val
        X_train, X_val, y_train, y_val = train_test_split(
            X_trainval, y_trainval, test_size=val_prop, random_state=random_state, stratify=stratify_trainval
        )

        return X_train, X_val, X_test, y_train, y_val, y_test


    def calibrate(self, fit_data_scaled, fit_labels):  
        """Calibrate the model using the specified method"""
        print("\n=== CALIBRATION ===")
        self.X_train, self.X_val, self.X_test, self.y_train, self.y_val, self.y_test = self._split_train_val_test(X=fit_data_scaled,
                                                                                                                  y=fit_labels)
        print(f"Training set size: {self.X_train.shape[0]} samples")
        print(f"Validation set size: {self.X_val.shape[0]} samples")

        if self.method == "isotonic":
            self.calibrated_model = CalibratedClassifierCV(estimator=self.uncalibrated_model, method='isotonic')
        elif self.method == "sigmoid":
            self.calibrated_model = CalibratedClassifierCV(estimator=self.uncalibrated_model, method="sigmoid", cv=self.cv)
        else:
            raise ValueError(f"Unknown calibration method: {self.method}")

        self.calibrated_model.fit(self.X_train, self.y_train)
        return self.calibrated_model
    
    
    def evaluate_calibrated_model_against_uncalibrated(self, test_data_scaled, test_labels):
        """Evaluate both calibrated and uncalibrated models on the test set"""
        print("\n=== EVALUATION: Calibrated vs Uncalibrated ===")
        if self.calibrated_model is None:
            raise ValueError("Model must be calibrated before evaluation.")

        # Uncalibrated model predictions
        y_prob_uncal = self.uncalibrated_model.predict_proba(self.X_test)[:, 1]
        y_pred_uncal = self.uncalibrated_model.predict(self.X_test)

        # Calibrated model predictions
        y_prob_cal = self.calibrated_model.predict_proba(self.X_test)[:, 1]
        y_pred_cal = self.calibrated_model.predict(self.X_test)

        self.uncalibrated_metrics = {
            'brier_score': brier_score_loss(self.y_test, y_prob_uncal),
            'log_loss': log_loss(self.y_test, y_prob_uncal),
            'roc_auc': roc_auc_score(self.y_test, y_prob_uncal),
            'accuracy': accuracy_score(self.y_test, y_pred_uncal),
            'precision': precision_score(self.y_test, y_pred_uncal),
            'recall': recall_score(self.y_test, y_pred_uncal),
            'f1_score': f1_score(self.y_test, y_pred_uncal)
        }
        self.calibrated_metrics = {
            'brier_score': brier_score_loss(self.y_test, y_prob_cal),
            'log_loss': log_loss(self.y_test, y_prob_cal),
            'roc_auc': roc_auc_score(self.y_test, y_prob_cal),
            'accuracy': accuracy_score(self.y_test, y_pred_cal),
            'precision': precision_score(self.y_test, y_pred_cal),
            'recall': recall_score(self.y_test, y_pred_cal),
            'f1_score': f1_score(self.y_test, y_pred_cal)
        }

        # Print metrics for both models

        print("\n--- Uncalibrated Model ---")
        print(f"Brier Score: {self.uncalibrated_metrics['brier_score']:.4f}")
        print(f"Log Loss: {self.uncalibrated_metrics['log_loss']:.4f}")
        print(f"ROC AUC: {self.uncalibrated_metrics['roc_auc']:.4f}")
        print(f"Accuracy: {self.uncalibrated_metrics['accuracy']:.4f}")
        print(f"Precision: {self.uncalibrated_metrics['precision']:.4f}")
        print(f"Recall: {self.uncalibrated_metrics['recall']:.4f}")
        print(f"F1 Score: {self.uncalibrated_metrics['f1_score']:.4f}")

        print("\n--- Calibrated Model ---")
        print(f"Brier Score: {self.calibrated_metrics['brier_score']:.4f}")
        print(f"Log Loss: {self.calibrated_metrics['log_loss']:.4f}")
        print(f"ROC AUC: {self.calibrated_metrics['roc_auc']:.4f}")
        print(f"Accuracy: {self.calibrated_metrics['accuracy']:.4f}")
        print(f"Precision: {self.calibrated_metrics['precision']:.4f}")
        print(f"Recall: {self.calibrated_metrics['recall']:.4f}")
        print(f"F1 Score: {self.calibrated_metrics['f1_score']:.4f}")

        print("\nClassification Report - Uncalibrated Model:")
        print(classification_report(self.y_test, y_pred_uncal))
        print("Classification Report - Calibrated Model:")
        print(classification_report(self.y_test, y_pred_cal))

        print("\nConfusion Matrix - Uncalibrated Model:")
        print(confusion_matrix(self.y_test, y_pred_uncal))
        print("\nConfusion Matrix - Calibrated Model:")
        print(confusion_matrix(self.y_test, y_pred_cal))

        print("Difference in metrics (Calibrated - Uncalibrated):")
        for key in self.calibrated_metrics:
            diff = self.calibrated_metrics[key] - self.uncalibrated_metrics[key]
            print(f"  {key}: {diff:.4f}")

        if self.visualize_results:
            fig = plt.figure(figsize=(20, 12))

            # 1. Calibration Curve (Reliability Diagram)
            ax1 = plt.subplot(2, 4, 1)
            fraction_of_positives_uncal, mean_predicted_value_uncal = calibration_curve(
                self.y_test, y_prob_uncal, n_bins=10, strategy='uniform'
            )
            fraction_of_positives_isotonic, mean_predicted_value_isotonic = calibration_curve(
                self.y_test, y_prob_cal, n_bins=10, strategy='uniform'
            )

            plt.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated', linewidth=2)
            plt.plot(mean_predicted_value_uncal, fraction_of_positives_uncal, 
                    's-', label='Uncalibrated', markersize=8, linewidth=2, color='red')
            plt.plot(mean_predicted_value_isotonic, fraction_of_positives_isotonic, 
                    'o-', label=f'{self.method}', markersize=8, linewidth=2, color='blue')
        
            plt.xlabel('Mean Predicted Probability', fontsize=11)
            plt.ylabel('Fraction of Positives', fontsize=11)
            plt.title('Calibration Curve (Reliability Diagram)', fontsize=12, fontweight='bold')
            plt.legend(loc='upper left')
            plt.grid(True, alpha=0.3)

            # 2. Distribuzione delle probabilità - Uncalibrated
            ax2 = plt.subplot(2, 4, 2)
            plt.hist(y_prob_uncal[self.y_test == 0], bins=30, alpha=0.5, 
                    label='Class 0', color='red', density=True)
            plt.hist(y_prob_uncal[self.y_test == 1], bins=30, alpha=0.5, 
                    label='Class 1', color='blue', density=True)
            plt.xlabel('Predicted Probability', fontsize=11)
            plt.ylabel('Density', fontsize=11)
            plt.title('Uncalibrated Distribution', fontsize=12, fontweight='bold')
            plt.legend()
            plt.grid(True, alpha=0.3)

            # 3. Distribuzione delle probabilità - Calibrated
            ax3 = plt.subplot(2, 4, 3)
            plt.hist(y_prob_cal[self.y_test == 0], bins=30, alpha=0.5, 
                    label='Class 0', color='red', density=True)
            plt.hist(y_prob_cal[self.y_test == 1], bins=30, alpha=0.5, 
                    label='Class 1', color='blue', density=True)
            plt.xlabel('Predicted Probability', fontsize=11)
            plt.ylabel('Density', fontsize=11)
            plt.title(f'{self.method} Calibration Distribution', fontsize=12, fontweight='bold')
            plt.legend()
            plt.grid(True, alpha=0.3)

            # 6. Scatter plot: Uncalibrated vs Sigmoid
            ax6 = plt.subplot(2, 4, 6)
            plt.scatter(y_prob_uncal, y_prob_cal, alpha=0.3, s=10, color='green')
            plt.plot([0, 1], [0, 1], 'r--', label='No change', linewidth=2)
            plt.xlabel('Uncalibrated Probability', fontsize=11)
            plt.ylabel(f'{self.method} Calibrated', fontsize=11)
            plt.title(f'{self.method} Transformation', fontsize=12, fontweight='bold')
            plt.legend()
            plt.grid(True, alpha=0.3)

            # 7. Difference in probabilities
            ax7 = plt.subplot(2, 4, 7)
            prob_diff_isotonic = y_prob_cal - y_prob_uncal
            plt.hist(prob_diff_isotonic, bins=50, edgecolor='black', alpha=0.7, color='blue')
            plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No change')
            plt.xlabel(f'{self.method} - Uncalibrated', fontsize=11)
            plt.ylabel('Frequency', fontsize=11)
            plt.title(f'{self.method} Adjustment Distribution', fontsize=12, fontweight='bold')
            plt.legend()
            plt.grid(True, alpha=0.3)

            

            plt.tight_layout()
            plt.savefig('calibration_comparison_full.png', dpi=300, bbox_inches='tight')
            plt.show()

            fig2 = plt.figure(figsize=(14, 6))

            # Binned comparison
            bins = np.linspace(0, 1, 11)
            bin_indices = np.digitize(y_prob_uncal, bins) - 1
            bin_means_uncal = [y_prob_uncal[bin_indices == i].mean() 
                            for i in range(len(bins)-1) if (bin_indices == i).sum() > 0]
            bin_means_isotonic = [y_prob_cal[bin_indices == i].mean() 
                                for i in range(len(bins)-1) if (bin_indices == i).sum() > 0]
            bin_means_sigmoid = [y_prob_cal[bin_indices == i].mean() 
                                for i in range(len(bins)-1) if (bin_indices == i).sum() > 0]
            bin_true = [pd.Series(self.y_test).iloc[bin_indices == i].mean() 
                        for i in range(len(bins)-1) if (bin_indices == i).sum() > 0]

            x_pos = np.arange(len(bin_means_uncal))
            width = 0.2

            ax = plt.subplot(1, 1, 1)
            plt.bar(x_pos - 1.5*width, bin_means_uncal, width, label='Uncalibrated', alpha=0.8, color='red')
            plt.bar(x_pos - 0.5*width, bin_means_isotonic, width, label=str(self.method), alpha=0.8, color='blue')
            plt.bar(x_pos + 1.5*width, bin_true, width, label='True Fraction', alpha=0.8, color='orange')
            plt.xlabel('Probability Bin', fontsize=11)
            plt.ylabel('Mean Probability', fontsize=11)
            plt.title('Binned Calibration Comparison', fontsize=12, fontweight='bold')
            plt.legend()
            plt.grid(True, alpha=0.3, axis='y')

            plt.tight_layout()
            plt.savefig('calibration_binned_comparison.png', dpi=300, bbox_inches='tight')
            plt.show()

        return self.uncalibrated_metrics, self.calibrated_metrics
    
    

c:\Users\atogni\anaconda3\envs\geok_5090\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
RAW_DATA_DIR = Path("E:/data")  
os.environ["RAW_DATA_DIR"] = str(RAW_DATA_DIR)

In [4]:
# Load data from Parquet
parquet_path = f"{RAW_DATA_DIR}/balanced_df_enh_encoder_10M.parquet"
df_parquet = pd.read_parquet(parquet_path)
fit_data_full = df_parquet.drop(columns=['label']).values
fit_labels_full = df_parquet['label'].values

#Load test data from Parquet
parquet_path_test = f"{RAW_DATA_DIR}/test_df_enh_encoder_2M.parquet"
df_parquet_test = pd.read_parquet(parquet_path_test)
test_data = df_parquet_test.drop(columns=['label']).values
test_labels = df_parquet_test['label'].values

In [ ]:
calibrator = XGboostCalibrator(
    model_path="E:/models/xgboost_compressed_data_dim20_model.joblib",
    scaler_path="E:/models/xgboost_scaler.joblib",
    fit_data=fit_data_full,
    fit_labels=fit_labels_full,
    test_data=test_data,
    test_labels=test_labels,
    method="sigmoid",
    visualize_results=True
)
calibrated_model = calibrator.calibrate(calibrator.fit_data_scaled, calibrator.fit_labels)
uncalibrated_metrics, calibrated_metrics = calibrator.evaluate_calibrated_model_against_uncalibrated(calibrator.test_data_scaled, calibrator.test_labels)